In [3]:
pip install albumentations opencv-python tqdm scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

import albumentations as A
from albumentations.pytorch import ToTensorV2


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [6]:
DATA_ROOT = "KolektorSDD-boxes"

image_paths = []
mask_paths = []

for kos_folder in sorted(os.listdir(DATA_ROOT)):
    kos_path = os.path.join(DATA_ROOT, kos_folder)

    if os.path.isdir(kos_path):
        images = glob(os.path.join(kos_path, "*.jpg"))

        for img_path in images:
            mask_path = img_path.replace(".jpg", "_label.bmp")

            if os.path.exists(mask_path):
                image_paths.append(img_path)
                mask_paths.append(mask_path)

print("Total samples:", len(image_paths))


Total samples: 399


In [7]:
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_imgs))
print("Val:", len(val_imgs))


Train: 319
Val: 80


In [8]:
class KSDDDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = cv2.imread(self.images[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.masks[idx], 0)

        label = 1 if mask.sum() > 0 else 0

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        mask = mask.unsqueeze(0).float() / 255.0

        return image, mask, torch.tensor(label).long()


In [9]:
train_tfms = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(),
    ToTensorV2()
])

val_tfms = A.Compose([
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])


In [10]:
train_dataset = KSDDDataset(train_imgs, train_masks, train_tfms)
val_dataset = KSDDDataset(val_imgs, val_masks, val_tfms)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


In [11]:
class CAMDenseNet(nn.Module):
    def __init__(self):
        super().__init__()

        base = models.densenet121(pretrained=True)
        self.features = base.features

        self.classifier = nn.Linear(1024, 2)

        self.seg_head = nn.Sequential(
            nn.Conv2d(1024, 512, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 1, 1)
        )

    def forward(self, x):
        feat = self.features(x)
        feat = F.relu(feat)

        pooled = F.adaptive_avg_pool2d(feat, 1).view(feat.size(0), -1)
        logits = self.classifier(pooled)

        seg_map = self.seg_head(feat)
        seg_map = F.interpolate(seg_map, size=x.shape[2:], mode="bilinear")

        return logits, seg_map, feat


In [12]:
def generate_cam(features, classifier_weight, labels):
    B, C, H, W = features.shape
    cams = []

    for i in range(B):
        weight = classifier_weight[labels[i]]
        cam = torch.sum(weight[:, None, None] * features[i], dim=0)
        cam = F.relu(cam)
        cam = (cam - cam.min()) / (cam.max() + 1e-6)
        cams.append(cam)

    return torch.stack(cams)


In [18]:
bce = nn.BCEWithLogitsLoss()
ce = nn.CrossEntropyLoss()

def cam_guided_loss(seg_pred, cam):
    cam = cam.unsqueeze(1)

    # Upsample CAM to match segmentation size
    cam = F.interpolate(
        cam,
        size=seg_pred.shape[2:],
        mode="bilinear",
        align_corners=False
    )

    return F.mse_loss(torch.sigmoid(seg_pred), cam)



In [19]:
model = CAMDenseNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [20]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, masks, labels in tqdm(loader):
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)

        logits, seg_pred, feat = model(images)

        cls_loss = ce(logits, labels)
        seg_loss = bce(seg_pred, masks)

        cams = generate_cam(feat, model.classifier.weight, labels)
        cam_loss = cam_guided_loss(seg_pred, cams)

        loss = cls_loss + seg_loss + 0.5 * cam_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [21]:
def validate(model, loader):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for images, masks, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits, _, _ = model(images)
            preds = logits.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [22]:
for epoch in range(10):
    train_loss = train_one_epoch(model, train_loader)
    val_acc = validate(model, val_loader)

    print(f"Epoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Val Accuracy:", val_acc)


100%|██████████| 40/40 [00:25<00:00,  1.59it/s]


Epoch 1
Train Loss: 0.5218662433326244
Val Accuracy: 0.8875


100%|██████████| 40/40 [00:12<00:00,  3.29it/s]


Epoch 2
Train Loss: 0.2732197569683194
Val Accuracy: 0.9625


100%|██████████| 40/40 [00:11<00:00,  3.34it/s]


...

Epoch 8
Train Loss: 0.03994441241957247
Val Accuracy: 0.975


100%|██████████| 40/40 [00:11<00:00,  3.36it/s]


Epoch 9
Train Loss: 0.029421834694221615
Val Accuracy: 0.9875


100%|██████████| 40/40 [00:11<00:00,  3.41it/s]


Epoch 10
Train Loss: 0.030182479228824378
Val Accuracy: 0.9875


In [23]:
torch.save(model.state_dict(), "cam_multitask_densenet_ksdd.pth")
print("Model saved successfully!")


Model saved successfully!
